## Exampe 1 - 간단한 신경망을 만들고, 결과를 그래프로 그리기

Copyright 2024, Joshua Starmer, 편집 및 수정: YoungDon Choi (K-water Research Institute)

### 노트북 개요

> **📌 이 Notebook에서 다룰 내용은**
> 1. **[사전 학습된 가중치(Weight)와 편향(Bias)을 가진 신경망 코딩하기](#create)** — `nn.Module`을 상속받는 클래스의 기본 구조와, `torch.tensor()`를 이용해 미리 학습된 가중치·편향 값을 만드는 방법을 보여줍니다.
> 2. **[신경망에 데이터 흘려보내기](#run)** — 이를 신경망의 **순전파(forward pass)** 라고 부릅니다. 입력 값을 신경망에 넣어 출력 값을 계산하는 과정입니다.
> 3. **[신경망 각 단계의 y축 값을 그래프로 그리기](#draw)** — 활성화 함수(activation function)에서 나온 y축 좌표들이 어떻게 스케일(scale) 조정되고 더해져서, 데이터에 맞는 구부러진(bent) 모양이 만들어지는지 시각적으로 보여줍니다.

<img src="images/chapter_1_pre_trained_nn.png" alt="a pre-trained neural network" style="width: 800px;">

### 신경망을 만들고 학습시키는 데 필요한 파이썬 모듈(module, 라이브러리)들을 불러오는(import) 단계

> 이번 노트북 전체에서 사용할 라이브러리를 불러오는(import) 코드입니다.
>
> - `import torch` — **PyTorch**의 핵심 모듈입니다. 텐서(tensor, 다차원 배열)를 만들고 다루기 위한 각종 함수를 제공합니다.
> - `import torch.nn as nn` — 신경망을 만들 때 필요한 `nn.Module`(신경망 클래스의 기반이 되는 클래스), `nn.Embedding()`, `nn.Linear()` 등을 제공합니다. 이 노트북에서는 `nn.Module`을 상속받아 신경망 클래스를 만듭니다.
> - `import torch.nn.functional as F` — `relu()`와 같은 활성화 함수들을 제공합니다. 뒤에서 ReLU 함수를 `F.relu()` 형태로 사용합니다.
> - `import matplotlib.pyplot as plt` — 그래프를 그리기 위한 기본 라이브러리입니다.
> - `import seaborn as sns` — matplotlib을 기반으로 좀 더 예쁘고 보기 좋은 그래프를 쉽게 그릴 수 있게 해주는 라이브러리입니다.

In [ ]:
import torch ## torch let's us create tensors and also provides helper functions
import torch.nn as nn ## torch.nn gives us nn.Module(), nn.Embedding() and nn.Linear()
import torch.nn.functional as F # This gives us relu()

import matplotlib.pyplot as plt ## matplotlib allows us to draw graphs.
import seaborn as sns ## seaborn makes it easier to draw nice-looking graphs.

### 사전학습된 신경망 만들기

> `nn.Module`을 상속받는 파이썬 클래스로 직접 만들어 볼 예정입니다. `nn.Module`을 상속받는 클래스는 보통 최소한 아래 두 개의 메서드(method)를 가집니다.
>
> - **`__init__()` 메서드**: 신경망을 구성하는 여러 요소(가중치, 편향 등)를 초기화(생성)하는 곳입니다.
> - **`forward()` 메서드**: 실제로 입력 값을 신경망에 통과시켜 계산을 수행하는 곳입니다. 즉, 데이터가 신경망을 따라 "앞으로(forward)" 흘러가는 과정을 정의합니다.
>
> 아래 이미지는 이 신경망에서 사용할 가중치(Weight)와 편향(Bias)에 각각 어떤 이름(w1, b1, w2, b2, w3, w4)을 붙였는지 보여줍니다. 이 이름들은 바로 다음 코드 셀에서 그대로 사용됩니다.

<img src="images/chapter_1_pre_trained_nn_labeled.png" alt="a pre-trained neural network with labeled weights and biases" style="width: 800px;">

> 신경망 클래스 `myNN`을 정의하는 핵심 코드입니다. `nn.Module`을 상속받고 있으며, 크게 두 부분으로 구성됩니다.
>
> **1) `__init__()` 메서드 — 가중치와 편향 초기화**
> - 클래스로부터 객체(object)를 만들 때(즉, `myNN()`을 호출할 때) 자동으로 실행되는 부분입니다.
> - `super().__init__()`은 부모 클래스인 `nn.Module`의 초기화 메서드를 먼저 호출하는 코드입니다. 상속을 받았다면 반드시 호출해줘야 부모 클래스가 제공하는 기능을 제대로 사용할 수 있습니다.
> - 이어서 `torch.tensor()`를 이용해 이미 학습이 끝난(pre-trained) 가중치와 편향 값을 하나씩 만들어 저장합니다.
>   - `w1 = 1.43`, `b1 = -0.61` → 입력에서 **위쪽(top)** 활성화 함수로 가는 연결의 가중치와 편향
>   - `w2 = 2.63`, `b2 = -0.27` → 입력에서 **아래쪽(bottom)** 활성화 함수로 가는 연결의 가중치와 편향
>   - `w3 = -3.89` → 위쪽 활성화 함수의 출력에 곱해지는 최종 가중치
>   - `w4 = 1.35` → 아래쪽 활성화 함수의 출력에 곱해지는 최종 가중치
>
> **2) `forward()` 메서드 — 순전파(데이터가 흘러가는 계산 과정)**
> - 이 클래스로 만든 객체에 입력 값을 넣으면(`model(input_values)`) 자동으로 호출되는 메서드입니다.
> - `top_x_axis_values = (input_values * self.w1) + self.b1` — 입력 값에 w1을 곱하고 b1을 더해서, "직선"을 하나 만듭니다(활성화 함수에 들어가기 전 값).
> - `bottom_x_axis_values = (input_values * self.w2) + self.b2` — 같은 방식으로 아래쪽 경로의 직선 값을 만듭니다.
> - `top_y_axis_values = F.relu(top_x_axis_values)` / `bottom_y_axis_values = F.relu(bottom_x_axis_values)` — 각각의 값을 **ReLU** 활성화 함수에 통과시킵니다. ReLU는 음수는 모두 0으로 만들고, 양수는 그대로 통과시켜서 "구부러진(bent)" 형태를 만듭니다.
> - `output_values = (top_y_axis_values * self.w3) + (bottom_y_axis_values * self.w4)` — 두 활성화 함수의 출력값에 각각 최종 가중치(w3, w4)를 곱한 뒤 더해서 최종 출력을 계산합니다.
> - 마지막으로 `return output_values`로 최종 결과를 반환합니다.
>
> 정리하면, 이 신경망은 **입력 → (2개의 선형 변환) → (2개의 ReLU 활성화 함수) → (가중치 곱해서 합산)** 순서로 계산되는, 은닉층(hidden layer) 노드가 2개인 아주 단순한 신경망입니다.

In [ ]:
class myNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.w1 = torch.tensor(1.43)
        self.b1 = torch.tensor(-0.61)

        self.w2 = torch.tensor(2.63)
        self.b2 = torch.tensor(-0.27)

        self.w3 = torch.tensor(-3.89)
        self.w4 = torch.tensor(1.35)


    def forward(self, input_values):

        top_x_axis_values = (input_values * self.w1) + self.b1
        bottom_x_axis_values = (input_values * self.w2) + self.b2

        top_y_axis_values = F.relu(top_x_axis_values)
        bottom_y_axis_values = F.relu(bottom_x_axis_values)

        output_values = (top_y_axis_values * self.w3) + (bottom_y_axis_values * self.w4)

        return output_values

### 데이터를 입력하여 신경망 결과(순전파: foward pass) 확인

> `myNN` 클래스의 인스턴스(객체)를 하나 생성하고, 관례적으로 `model`이라는 변수 이름에 저장합니다. 이 시점에서 `__init__()` 메서드가 실행되어, w1~w4, b1, b2 값들이 모두 초기화됩니다.

In [ ]:
model = myNN()

> 신경망에 통과시킬 입력 값들을 텐서(tensor)로 만듭니다. 여기서는 약물 투여량(Dose)을 의미하는 값 `0.0`, `0.5`, `1.0`을 예시로 사용합니다.
>
> `model(doses)`처럼 모델 객체를 함수처럼 호출하면, PyTorch가 내부적으로 `model.forward(doses)`를 자동으로 실행해줍니다. 즉, 직접 `forward()`를 호출하지 않아도 됩니다. 이 셀을 실행하면 각 입력값(dose)에 대한 신경망의 출력값이 계산되어 화면에 표시됩니다.

In [ ]:
## Create a tensor with input doses
doses = torch.tensor([0.0, 0.5, 1.0])

## NOTE: By default, the forward() method will be called on the input
model(doses)

> 위 셀에서 나온 출력값은 소수점 아래 자리수가 길어서 읽기 불편할 수 있습니다. `torch.round(..., decimals=2)`를 사용해 결과를 소수점 둘째 자리까지 반올림하여, 더 읽기 쉬운 형태로 다시 출력합니다.

In [ ]:
## we can use torch.round() to round it to the nearest 100th.
torch.round(model(doses), decimals=2)

### 신경망 내부에서 계산이 진행되는 **각 단계**의 값들을 그래프로 그려보기

> 그래프를 그리기에 앞서, 신경망의 구조(가중치·편향 이름 포함)를 다시 한 번 이미지로 확인합니다. 아래에서 계산할 `w1, b1, w2, b2, w3, w4`가 신경망의 어느 연결에 해당하는지 시각적으로 참고할 수 있습니다.

First, let's remember what the neural network looks like...
<img src="images/chapter_1_pre_trained_nn_labeled.png" alt="a pre-trained neural network" style="width: 800px;">

### 신경망의 **위쪽(top)** 경로에 있는 활성화 함수(ReLU)의 출력값을, 입력값(dose)에 대한 그래프

> 그래프를 그리기 위해 다양한 입력값(dose)을 만듭니다. `torch.linspace(start=0, end=1, steps=11)`은 0부터 1까지의 구간을 **11개의 값**으로 균등하게 나눠서 만들어줍니다(0, 0.1, 0.2, ... 1.0). 이렇게 만든 값들을 `input_doses`에 저장하고, 마지막 줄에서 실제로 어떤 값들이 만들어졌는지 출력해서 확인합니다.

In [ ]:
input_doses = torch.linspace(start=0, end=1, steps=11)
input_doses

> 입력값들(`input_doses`)에 위쪽 경로의 가중치 `w1`을 곱하고 편향 `b1`을 더해서, ReLU에 들어가기 직전의 x축 값을 계산합니다. 이는 신경망 그림에서 "입력 → 위쪽 활성화 함수"로 이어지는 연결에 해당하는 선형 변환입니다.

In [ ]:
top_x_axis_values = (model.w1 * input_doses) + model.b1
top_x_axis_values

> 앞서 계산한 `top_x_axis_values`를 `F.relu()`에 통과시켜 **ReLU 활성화 함수**의 출력값(y축 값)을 계산합니다. ReLU는 입력이 0보다 작으면 0으로, 0 이상이면 입력값을 그대로 반환하는 함수입니다. 이 결과가 그래프에서 꺾인(구부러진) 모양을 만드는 핵심 요소입니다.

In [ ]:
## now run those x-axis values through the ReLU...
top_y_axis_values = F.relu(top_x_axis_values)
top_y_axis_values

> `input_doses`(x축)와 `top_y_axis_values`(y축)를 이용해 그래프를 그립니다.
>
> - `sns.set(style="whitegrid")` — seaborn 그래프의 배경 스타일을 흰색 격자무늬로 설정합니다.
> - `sns.scatterplot(...)` — 각 데이터 포인트를 파란색(`color='blue'`) 점으로 찍습니다.
> - `sns.lineplot(...)` — 그 점들을 선으로 연결해서 이어줍니다.
> - `plt.ylabel(...)`, `plt.xlabel(...)` — y축은 "Upper ReLU Output"(위쪽 ReLU 출력), x축은 "Dose"(투여량)로 라벨을 붙입니다.
>
> 결과적으로 ReLU를 거친 위쪽 경로의 출력이 dose가 커짐에 따라 어떻게 꺾인 모양으로 변하는지 확인할 수 있습니다.

In [ ]:
## First, set the style for seaborn so that the graph looks cool.
sns.set(style="whitegrid")

## First, draw the individual points
sns.scatterplot(x=input_doses,
                y=top_y_axis_values,
                color='blue',
                s=200)

## Now connect those points with a line
sns.lineplot(x=input_doses,
             y=top_y_axis_values,
             color='blue',
             linewidth=2.5)

## now label the y- and x-axes.
plt.ylabel('Upper ReLU Output')
plt.xlabel('Dose')

### 신경망의 **아래쪽(bottom)** 경로에 있는 활성화 함수의 출력값을, 입력값(dose)에 대한 그래프

> 입력값들에 아래쪽 경로의 가중치 `w2`를 곱하고 편향 `b2`를 더해서, ReLU에 들어가기 직전의 x축 값을 계산합니다. 위쪽 경로(셀 20)와 계산 방식은 동일하지만 사용하는 가중치·편향이 다릅니다.

In [ ]:
bottom_x_axis_values = (model.w2 * input_doses) + model.b2
bottom_x_axis_values

> `bottom_x_axis_values`를 `F.relu()`에 통과시켜 아래쪽 경로의 ReLU 출력값(y축 값)을 계산합니다.

In [ ]:
bottom_y_axis_values = F.relu(bottom_x_axis_values)
bottom_y_axis_values

위쪽 경로 그래프(셀 22)와 동일한 방식으로, 이번에는 아래쪽 경로의 결과를 그래프로 그립니다. 색상만 주황색(`color='orange'`)으로 다르게 설정하고, y축 라벨은 "Bottom ReLU Output"(아래쪽 ReLU 출력)으로 지정합니다.

In [ ]:
## First, set the style for seaborn so that the graph looks cool.
sns.set(style="whitegrid")

## First, draw the individual points
sns.scatterplot(x=input_doses,
                y=bottom_y_axis_values,
                color='orange',
                s=200)

## Now connect those points with a line
sns.lineplot(x=input_doses,
             y=bottom_y_axis_values,
             color='orange',
             linewidth=2.5)

## now label the y- and x-axes.
plt.ylabel('Bottom ReLU Output')
plt.xlabel('Dose')

### 앞서 따로따로 그렸던 위쪽 경로와 아래쪽 경로의 ReLU 출력을 **하나의 그래프 위에 함께** 그려서 두 곡선을 비교

> 하나의 그래프 위에 위쪽 경로(파란색)와 아래쪽 경로(주황색)의 ReLU 출력을 모두 그립니다. 각각 산점도(scatterplot)로 점을 찍고, 선(lineplot)으로 이어준 뒤, y축 라벨을 "ReLU Outputs"로 설정합니다. 이렇게 하면 두 개의 꺾인 직선(구부러진 모양)이 서로 다른 지점에서 꺾이는 것을 한눈에 비교할 수 있습니다.

In [ ]:
## First, set the style for seaborn so that the graph looks cool.
sns.set(style="whitegrid")

## Draw the individual points (top)
sns.scatterplot(x=input_doses,
                y=top_y_axis_values,
                color='blue',
                s=200)

## Connect those points with a line (top)
sns.lineplot(x=input_doses,
             y=top_y_axis_values,
             color='blue',
             linewidth=2.5)

## Draw the individual points (bottom)
sns.scatterplot(x=input_doses,
                y=bottom_y_axis_values,
                color='orange',
                s=200)

## Connect those points with a line (bottom)
sns.lineplot(x=input_doses,
             y=bottom_y_axis_values,
             color='orange',
             linewidth=2.5)

## now label the y- and x-axes.
plt.ylabel('ReLU Outputs')
plt.xlabel('Dose')

### 두 개의 구부러진 모양(bent shape)을, 신경망의 마지막 가중치(`w3`, `w4`)로 **스케일(크기) 조정**

> 위쪽 경로의 ReLU 출력값(`top_y_axis_values`)에 최종 가중치 `w3`(값: -3.89)를 곱합니다. `w3`가 음수이기 때문에, 이 연산을 거치면 그래프의 모양이 위아래로 뒤집히면서 동시에 크기가 늘어납니다(스케일 조정).

In [ ]:
final_top_y_axis_values = top_y_axis_values * model.w3
final_top_y_axis_values

> 아래쪽 경로의 ReLU 출력값(`bottom_y_axis_values`)에 최종 가중치 `w4`(값: 1.35)를 곱합니다. `w4`는 양수이므로 그래프 모양은 뒤집히지 않고, 세로 크기만 커집니다.

In [ ]:
final_bottom_y_axis_values = bottom_y_axis_values * model.w4
final_bottom_y_axis_values

> `w3`, `w4`로 스케일 조정된 두 곡선(`final_top_y_axis_values`, `final_bottom_y_axis_values`)을 하나의 그래프에 함께 그립니다. 앞선 그래프(셀 28)와 그리는 방식은 동일하지만, 이번에는 최종 가중치가 곱해진 후의 값을 사용하므로 곡선의 모양(방향과 크기)이 달라진 것을 확인할 수 있습니다. y축 라벨은 "Final Bent Shapes for Top and Bottom"으로 설정합니다.

In [ ]:
## First, set the style for seaborn so that the graph looks cool.
sns.set(style="whitegrid")

## Draw the individual points (top)
sns.scatterplot(x=input_doses,
                y=final_top_y_axis_values,
                color='blue',
                s=200)

## Connect those points with a line (top)
sns.lineplot(x=input_doses,
             y=final_top_y_axis_values,
             color='blue',
             linewidth=2.5)

## Draw the individual points (bottom)
sns.scatterplot(x=input_doses,
                y=final_bottom_y_axis_values,
                color='orange',
                s=200)

## Connect those points with a line (bottom)
sns.lineplot(x=input_doses,
             y=final_bottom_y_axis_values,
             color='orange',
             linewidth=2.5)

## now label the y- and x-axes.
plt.ylabel('Final Bent Shapes for Top and Bottom')
plt.xlabel('Dose')

### 마지막 단계로, 스케일 조정이 끝난 두 개의 구부러진 모양을 **서로 더해서** 신경망의 최종 출력

> `final_top_y_axis_values`와 `final_bottom_y_axis_values`를 요소별로 더해서 `final_bent_shape`를 계산합니다. 이 값이 바로 신경망의 `forward()` 메서드가 최종적으로 반환하는 출력값과 동일합니다(셀 7의 `output_values` 계산 마지막 줄과 같은 연산입니다). 즉, 신경망 전체를 통과한 최종 결과를 각 dose 값마다 확인할 수 있습니다.

In [ ]:
final_bent_shape = final_top_y_axis_values + final_bottom_y_axis_values
final_bent_shape

> 최종 결과값(`final_bent_shape`)을 초록색(`color='green'`)으로 그래프에 그립니다. 산점도와 선 그래프를 함께 그려서, dose(x축)가 변함에 따라 신경망의 최종 출력(y축)이 어떤 구부러진 곡선 형태를 이루는지 보여줍니다. 이 곡선이 바로 이 신경망이 데이터를 근사(fit)하기 위해 만들어낸 최종 함수 모양입니다. y축 라벨은 "Final Bent Shape"로 설정합니다.

In [ ]:
## Now put both bent shapes on the same graph...

## First, set the style for seaborn so that the graph looks cool.
sns.set(style="whitegrid")

## Draw the individual points
sns.scatterplot(x=input_doses,
                y=final_bent_shape,
                color='green',
                s=200)

## Connect those points with a line
sns.lineplot(x=input_doses,
             y=final_bent_shape,
             color='green',
             linewidth=2.5)

## now label the y- and x-axes.
plt.ylabel('Final Bent Shape')
plt.xlabel('Dose')